In [1]:
import torch
import pandas as pd
import tqdm
from torch.utils.data import DataLoader, Dataset

# =========================================================================
# 1. DEFINE A LIGHTWEIGHT TEST DATASET (No Label Lookup Required)
# =========================================================================
class GalaxyTestDataset(Dataset):
    """Safely handles test imagery where labels do not exist in the csv."""
    def __init__(self, base_dataset):
        self.images = base_dataset.images
        self.galaxy_ids = base_dataset.galaxy_ids
        self.transform = base_dataset.transform

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
            
        # Return a dummy label array of zeros so the DataLoader structure matches
        dummy_label = torch.zeros(37, dtype=torch.float32, device=img.device)
        return img, dummy_label

    def __len__(self):
        return len(self.images)

# =========================================================================
# 2. INITIALIZE AND WRAP THE DATA
# =========================================================================
# Create your standard test set object using your file path list
raw_test_set = GalaxyDataset(test_paths, labels_csv, transform=val_transform, cache_path='test_set.pth')

# Wrap it in our safe class to block the Pandas .loc lookups completely
safe_test_set = GalaxyTestDataset(raw_test_set)
test_loader = DataLoader(safe_test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS)

# =========================================================================
# 3. GENERATE PREDICTIONS
# =========================================================================
model.eval()
all_predictions = []

with torch.no_grad():
    for imgs, _ in tqdm.tqdm(test_loader, desc="Generating Predictions"):
        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            outputs = model(imgs)
        all_predictions.extend(outputs.cpu().float().numpy().tolist())

# =========================================================================
# 4. EXSTRUCT HEADERS AND SAVE
# =========================================================================
column_names = pd.read_csv(labels_csv, nrows=1).columns.tolist()
if 'GalaxyID' in column_names:
    column_names.remove('GalaxyID')

submission_df = pd.DataFrame(all_predictions, columns=column_names)
submission_df.insert(0, 'GalaxyID', safe_test_set.galaxy_ids)

output_path = r"C:\Users\harty\Code\Galaxy-Classifier\test_predictions.csv"
submission_df.to_csv(output_path, index=False)

print(f"★ Success! Prediction file generated with shape {submission_df.shape} at:\n{output_path}")


NameError: name 'GalaxyDataset' is not defined